In [ ]:
import os
# For a fair compiraison of performance against NumPy, we disable multithreading
os.environ["XLA_FLAGS"] = "--xla_cpu_multi_thread_eigen=false intra_op_parallelism_threads=1"
os.environ["OMP_NUM_THREADS"] = "1"

import jax

# For a fair comparaison of performance against NumPy, we use doubles for JAX computations
jax.config.update('jax_enable_x64', True)

import jax.numpy as jnp
import matplotlib.pyplot as plt

# Automatic differentiation

As with for many JAX features, automatic differentiation is performed through transformations.
There are many available (`grad`, `jvp`, `vjp`, ...) and we will explore some of those in this notebook.
However, before getting into this, it is best to start with some primer on [automatic differentiation](https://docs.jax.dev/en/latest/automatic-differentiation.html).

## Getting derivatives

To start off, there are basically three ways to compute the local derivative of some function:
1. [analytical (symbolic) differentiation](https://en.wikipedia.org/wiki/Differentiation_rules) ― compute derivate analytically
2. [numerical differentiation](https://en.wikipedia.org/wiki/Numerical_differentiation) ― perturb a function to estimate its local derivative 
3. [automatic differentiation](https://en.wikipedia.org/wiki/Automatic_differentiation#Forward_and_reverse_accumulation) ― today's topic!

Analytical (symbolic) differentiation (done by hand or with a CAS like SymPy) gives exact, closed-form derivatives, but it needs a closed-form functions (difficulty to encode arbitrary control flow), and the expressions grow exponentially as compositions deepen, duplicating shared subterms instead of reusing them.

Numerical differentiation is simple, but it scales terribly: each partial derivative costs one extra function evaluation, so a full gradient in $n$ dimensions needs $O(n)$ passes, which is basically hopeless for most real world applications (while we know today's frontier LLM reaches the trillion-parameter range!).

Automatic differentiation has none of those problems.
It breaks the function into elementary operations (including arbitrary control flow) and applies the chain rule mechanically to the values flowing through, reusing intermediates rather than recomputing or duplicating them.
The result is exact to machine precision at a cost that's a small multiple of evaluating the function itself.
All that's left to fix is the direction in which the chain rule is applied: either forward or reverse accumulation, which we will see shortly.
There is however one downside vs symbolic differentiation: you get the derivative only at the point you evaluate, never as a closed-form expression.
For most practical problems, this is a non issue.

## Automatic differentiation

More generally, the derivative of a function is not a number or a matrix, but a linear mapping.
For $f: \mathbb{R}^n \longrightarrow \mathbb{R}^m$, locally at a point $x$, the derivative $\partial f(x)$ is a linear map which best approximate $f$ around $x$,
$$f(x+\delta) \approx f(x) + \partial f(x) \delta.$$
Selecting a basis for this linear mapping, we get the Jacobian $J \in \mathbb{R}^{m\times n}$.
However, automatic differentiation is about computing the mapping and never evaluating the array directly.

Given a mapping $f: \mathbb{R}^n \longrightarrow \mathbb{R}^m$ and its Jacobian $J$, there are exactly two ways to contract it with a vector:

- **Jacobian-vector product (JVP)**: $Jv$, with $v \in \mathbb{R}^n$: contraction on the input side -> forward mode,
- **Vector-Jacobian product (VJP)**: $u^\top J$, with $u \in \mathbb{R}^m$: contraction on the output side -> reverse mode.

Each product returns a derivative only once seeded with a concrete vector: a basis seed picks out one column (JVP) or row (VJP) of $J$ (we won't do this by hand in this notebook as higher-level tools handle the seeding for us).

A single JVP and a single VJP cost the same: one sweep through $f$.
In forward mode, derivatives are simply carried alongside the primal (ordinary values of $f$) in a single pass.
On the other hand, reverse mode must record the entire forward computation to replay it backward (which includes its intermediate values).
As a result, reverse mode differentiation will have a memory footprint growing with the length of the computation, while forward mode stays flat.

Since computing the derivative is more efficient in forward mode, why would reverse mode be even useful?
Most optimization problems are about minimizing a scalar valued function.
For example, least square minimization is about minimizing the sum of the squared residuals of a model $f(x, \theta)$:
$$
\min_\theta \sum_{i=1}^{N} (f(x_i; \theta) - y_i)^2.
$$

This means that computing its gradient is done in one pass, starting from the scalar value result.
In forward mode, the same derivative computation would need as many passes as degree of freedom.
In practice, you will quasi exclusively use reverse mode differentiation.

More generally, for a given function
$$f: \mathbb{R}^n \longrightarrow \mathbb{R}^m,$$
it is faster (less evaluations) to derivate in forward mode when $n < m$ and reverse mode otherwise.
In practice, mostly because of the memory footprint, for problems where $n \approx m$, forward mode will be faster.

> _**Note:**_ The autodifferentiation engine in JAX is very powerfull and we're only touching its surface in this notebook.
> In particular, we are not exploring the `jax.vjp` and `jax.jvp` vector/jacobian products, which are powerfull and interesting building blocks.
> For those interested into getting deeper, follow this [link](https://docs.jax.dev/en/latest/notebooks/autodiff_cookbook.html) and related documentation.

> _**Note:**_ Some JAX primitives do not support reverse mode autodiff. One example is `jax.lax.while_loop`. One solution is to manually define the derivative rules, as shown [here](https://docs.jax.dev/en/latest/hijax_custom_derivatives.html).

# Scalar function derivatives

`grad` ([link to documentation](https://docs.jax.dev/en/latest/_autosummary/jax.grad.html), read it!) is a JAX transformation which computes the gradient of a scalar function through reverse mode.

## $f: \mathbb{R} \longrightarrow \mathbb{R}$

As a first example, we use `grad` on a simple $f: \mathbb{R} \longrightarrow \mathbb{R}$.
`grad` can also be composed to get higher order derivatives.

> _**Note:**_ As it can be seen, the derivative computed using AD is equal to its analytical formulation up to machine precision. AD is not an approximation.

> _**Note:**_ Since `grad` is a transform, it can more generally be applied to pytrees.

In [ ]:
def f(x):
    return jnp.sin(x)

def df_dx(x):
    return jnp.cos(x)

def d2f_dx2(x):
    return -jnp.sin(x)

In [ ]:
print("Analytical df_dx: ", df_dx(0.), ", AD: ", jax.grad(f)(0.))
print("Analytical d2f_dx2: ", d2f_dx2(1.), ", AD: ", jax.grad(jax.grad(f))(1.))

We now introduce `jacfwd` ([link documentation](https://docs.jax.dev/en/latest/_autosummary/jax.jacfwd.html)) and `jacrev` ([link to documentation](https://docs.jax.dev/en/latest/_autosummary/jax.jacrev.html)) which computes the jacobian either through forward or reverse AD.

In [ ]:
print("Analytical df_dx", df_dx(1.), ", AD (forward): ", jax.jacfwd(f)(1.))
print("Analytical df_dx", df_dx(1.), ", AD (reverse): ", jax.jacrev(f)(1.))

In the case of a $f:\mathbb{R} \longrightarrow \mathbb{R}$ function, their performance are similar:

In [ ]:
@jax.jit(static_argnums=0)
def df_fwd(f, x):
    return jax.jacfwd(f)(x)

@jax.jit(static_argnums=0)
def df_rev(f, x):
    return jax.jacrev(f)(x)

@jax.jit(static_argnums=0)
def df_grad(f, x):
    return jax.grad(f)(x)

In [ ]:
%timeit df_fwd(f, 0.)
%timeit df_rev(f, 0.)
%timeit df_grad(f, 0.)

## $f: \mathbb{R}^N \longrightarrow \mathbb{R}$

We now compute the gradient of some $f: \mathbb{R}^N \longrightarrow \mathbb{R}$ function, similar to what we would do for classical optimization.

In this case, reverse mode AD is much faster.

In [ ]:
# Some silly function which reduces into a scalar
def f(x):
    return jnp.sum(jnp.exp(-x**2)*x/4.+jnp.cos(2.*jnp.pi*x)**3-jnp.tan(x)**2*jnp.log10(jnp.sin(x))*5./jnp.cos(x+1/x))

In [ ]:
N = 100
x = jax.random.normal(jax.random.key(1337), N)

%timeit df_fwd(f, x).block_until_ready()
%timeit df_rev(f, x).block_until_ready()
%timeit df_grad(f, x).block_until_ready()

> _**Exercice:**_ Show that when the number of input dimensions is equal to the same number of output dimensions, forward mode is slightly faster than reverse mode.

In [ ]:
# Uncomment this to get the solution!
# %load solutions/ad_fwd_vs_rev.py

## `value_and_grad`
In most cases, particularly for optimization problems, we are interested into both the value of some scalar function and its derivative.
Since reverse mode AD needs a first forward pass to compute the residuals, we actually get its value for "free".

Usage:

In [ ]:
print(jax.value_and_grad(f)(1.))
print(f(1.), jax.grad(f)(1.))

Comparing performance of `value_and_grad` vs a naive implementation.

In [ ]:
def f_and_df_vag(x):
    return jnp.hstack(jax.value_and_grad(f)(x))

def f_and_df_naive(x):
    return jnp.hstack((f(x), jax.grad(f)(x)))

x = jax.random.normal(jax.random.key(1337), 10_000_000)
%timeit f(x).block_until_ready()
%timeit f_and_df_naive(x).block_until_ready()
%timeit f_and_df_vag(x).block_until_ready()

_**Question:**_ What happens when we jit compile `f_and_df_vag` and `f_and_df_naive`? Why would it be?

# Gradient descent
As a first application of automatic differentiation, lets implement a simple [gradient descent algorithm](https://en.wikipedia.org/wiki/Gradient_descent).

We define the 2D [Rosenbrock function](https://en.wikipedia.org/wiki/Rosenbrock_function) as

$$
\begin{align}
f: \quad & (\mathbb{R}^2; \mathbb{R}^2) & \longrightarrow & \mathbb{R} \\
& f(x, y; a, b) & \longmapsto & (a-x)^2+b(y-x^2)^2
\end{align}
$$

Let us first plot the function on the $x-y$ plane with $a=1, b=100$ (its minimum is then at $(1, 1)$).

In addition, as a nice application of `vmap`, we will for each sampled point on the plane evaluate its derivative and plot its direction on top.

In [ ]:
# Define Rosenbrock function
def rosenbrock_2d(x, a, b):
    return (a-x[0])**2+b*(x[1]-x[0]**2)**2

# Define plot region and Rosenbrock function parameters
a, b = 1., 100.
xmin, xmax = -3., 3.
ymin, ymax = -3., 3.
N = 20

# Build a point vector on which we will evaluate the Rosenbrock function
XY = jnp.stack(jnp.meshgrid(jnp.linspace(xmin, xmax, N), jnp.linspace(ymin, ymax, N))).reshape(2, -1).T

# Evaluate Rosenbrock function on the point vector
F = jax.vmap(lambda x: rosenbrock_2d(x, a, b))(XY).reshape(N, N)
UV = jax.vmap(jax.grad(lambda x: rosenbrock_2d(x, 1., 100.)))(XY)

# Lets plot
plt.figure(figsize=(5., 5.), layout='constrained')
plt.imshow(F, extent=(xmin, xmax, ymin, ymax), origin='lower')
plt.quiver(XY[:, 0], XY[:, 1], UV[:, 0], UV[:, 1], color='white')
plt.xlim(xmin, xmax)
plt.ylim(ymin, ymax)
#plt.axis('equal')
plt.show()

We now write some simple gradient descent algorithm.

Given a position $x_n$, the next position $x_{n+1}$ is given by

$$x_{n+1} = x_n - \eta \nabla f(x_n),$$

until some [convergence criterion](https://en.wikipedia.org/wiki/Convergence_tests) is fulfilled (in our case, we compare the Euclidean distance of the step size, $\sqrt{\sum \eta \nabla f(x_n)}$, to 0 using `jnp.isclose`).

In [ ]:
def minimize_gradient_descent(f, x_0, step_size=1., atol=1e-5, rtol=1e-5, maxiter=100):
    g = jax.grad(f)

    def _step(x):
        return -step_size*g(x)

    x = x_0 # Initial position
    s = _step(x) # Initial step
    h = [x] # Position history
    i = 0 # Loop count
    
    # We loop until convergence criterion is satisfied, we're running out of allowed itterations or the function becomes nan.
    while jnp.logical_not(jnp.isclose(jnp.sqrt(jnp.sum(s**2)), 0., atol=atol, rtol=rtol)) and i < maxiter and jnp.logical_not(jnp.any(jnp.isnan(s))):
        x = x + s
        s = _step(x)
        h.append(x+s)
        i += 1

    # Return found position and history
    return x, jnp.array(h)

In [ ]:
minimize_gradient_descent(lambda x: rosenbrock_2d(x, a, b), jnp.array([1.01, 1.01]), step_size=0.001, maxiter=10000)

> _**Exercice:**_ If you feel like it, rewrite the algorithm using `jax.while_loop` and jit it.

> _**Note:**_ As you can see, the Rosenbrock function is known to be a very difficult function to minimize. It will fail gradient descent if starting too far from the optimal point and with too big step size.
> To convince yourself, you can try this algorithm  on [simpler functions](https://en.wikipedia.org/wiki/Test_functions_for_optimization), for example on some convex ones.

> _**Exercice:**_ JAX also exposes a transformation which computes the Hessian matrix, `jax.hessian` ([link to documentation](https://docs.jax.dev/en/latest/_autosummary/jax.hessian.html), read it!).
> Implement 2nd order Newton's method optimization algorithm and test it on the Rosenbrock function.

In [ ]:
# Uncomment for solution!
# %load solutions/ad_newton_method.py

In practice, you will never implement the minimization methods yourself.
For this, there are two main libraries:
- [Optax](https://optax.readthedocs.io/en/latest/) ― Gradient based optimizers, best suited for large neural networks.
- [Optimistix](https://github.com/patrick-kidger/optimistix) ― General purpose optimizers, which implements higher order methods (CG, BFGS, ...). Can also interface Optax.

> _**Note:**_ [JAXopt](https://github.com/google/jaxopt), while somewhat standard in the past, is now discontinued and should not be used for new projects.

# Appendix: evaluate derivative of a function mapped on an axis

It can sometimes be usefull (for example, to do sensitivity analysis) to compute the derivative of some function against some variable mapped on an axis.
This is a prime application of `vmap`.

We will play with 1D polynomials as an example, since their analytical derivatives are trivial to compute.

In [ ]:
coeffs = jnp.array([-0.5, 1.5, 1.])
def f(x, theta):
    return jnp.polyval(theta, x)

def df_dx(x, theta):
    return jnp.polyval(jnp.polyder(theta), x)


We now vmap the derivative over $x$:

In [ ]:
def vgrad(f):
    return lambda x: jax.vmap(lambda i: jax.jacfwd(f)(x[i]))(jnp.arange(len(x)))

In [ ]:
x = jnp.linspace(0., 1.)
df_dx_ad = vgrad(lambda x: f(x, coeffs))

In [ ]:
plt.plot(x, f(x, coeffs), label="$f$")
plt.plot(x, df_dx(x, coeffs), label="$f'$ - Analytical")
plt.plot(x, df_dx_ad(x), label="$f'$ - AD")
plt.legend()
plt.grid()
plt.show()

In [ ]:
g_dg = lambda x, theta: jax.vmap(lambda i: jax.value_and_grad(g)(x[i], theta))(jnp.arange(len(x)))

In [ ]:
x = jnp.linspace(0., 1.)
plt.plot(x, g(x, coeffs))
plt.plot(x, dg_dx(x, coeffs))
plt.plot(x, ad_dg_dx_vmap(lambda x: g(x, coeffs)))
plt.show()